In [ ]:
%pip install -U -q transformers datasets accelerate scikit-learn "protobuf==3.20.3" sentencepiece

In [ ]:
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset, Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    TrainerCallback
)
import warnings
warnings.filterwarnings('ignore')

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = 'cuda' 

In [2]:
ds = load_dataset("ailsntua/QEvasion")

train_full_df = ds['train'].to_pandas()
test_df = ds['test'].to_pandas()

train_df, val_df = train_test_split(
    train_full_df,
    test_size=0.15,
    random_state=SEED,
    stratify=train_full_df['clarity_label']
)

print(f"Train: {len(train_df)}")
print(f"Validation: {len(val_df)}")
print(f"Test: {len(test_df)}")

Train: 2930
Validation: 518
Test: 308


In [ ]:
MODEL_NAME = "FacebookAI/roberta-large"
MAX_LENGTH = 512

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

label2id = {"Clear Reply": 0, "Ambivalent": 1, "Clear Non-Reply": 2}
id2label = {v: k for k, v in label2id.items()}
clarity_labels = ["Clear Reply", "Ambivalent", "Clear Non-Reply"]

In [4]:
def tokenize_function(examples):
    inputs = [
        f"Question: {q}\nAnswer: {a}"
        for q, a in zip(examples["question"], examples["interview_answer"])
    ]
    tokenized = tokenizer(
        inputs,
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False
    )
    tokenized["labels"] = [label2id[l] for l in examples["clarity_label"]]
    return tokenized

train_dataset = Dataset.from_pandas(train_df, preserve_index=False)
val_dataset = Dataset.from_pandas(val_df, preserve_index=False)

train_tokenized = train_dataset.map(tokenize_function, batched=True, remove_columns=train_dataset.column_names)
val_tokenized = val_dataset.map(tokenize_function, batched=True, remove_columns=val_dataset.column_names)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

Map:   0%|          | 0/2930 [00:00<?, ? examples/s]

Map:   0%|          | 0/518 [00:00<?, ? examples/s]

In [ ]:
test_dataset = Dataset.from_pandas(test_df, preserve_index=False)
test_tokenized = test_dataset.map(tokenize_function, batched=True, remove_columns=test_dataset.column_names)

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    accuracy = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average='macro')
    return {'accuracy': accuracy, 'f1': f1}

In [7]:
class EarlyStoppingCallback(TrainerCallback):
    def __init__(self, patience: int = 3, metric_name: str = "eval_f1", greater_is_better: bool = True):
        self.patience = patience
        self.metric_name = metric_name
        self.greater_is_better = greater_is_better
        self.best_metric = None
        self.patience_counter = 0
        
    def on_evaluate(self, args, state, control, metrics, **kwargs):
        current_metric = metrics.get(self.metric_name)
        
        if current_metric is None:
            return
        
        if self.best_metric is None:
            self.best_metric = current_metric
            self.patience_counter = 0
        else:
            if self.greater_is_better:
                improved = current_metric > self.best_metric
            else:
                improved = current_metric < self.best_metric
            
            if improved:
                self.best_metric = current_metric
                self.patience_counter = 0
            else:
                self.patience_counter += 1
                
        if self.patience_counter >= self.patience:
            print(f"\nEarly stopping triggered!")
            print(f"Best {self.metric_name}: {self.best_metric:.4f}")
            control.should_training_stop = True

In [ ]:
training_args = TrainingArguments(
    output_dir="./results_roberta_large",
    learning_rate=1e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=15,
    warmup_ratio=0.1,
    weight_decay=0.01,
    max_grad_norm=1.0,
    gradient_checkpointing=True,
    bf16=True,
    bf16_full_eval=True,
    optim="adamw_torch",
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    logging_steps=50,
    logging_strategy="steps",
    seed=SEED,
    report_to="none",
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(patience=3, metric_name="eval_f1")],
)

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    id2label=id2label,
    label2id=label2id
)

In [11]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.926500,0.912068,0.583012,0.246129
2,0.881000,0.884898,0.573359,0.334946
3,0.758400,0.729105,0.675676,0.636345
4,0.594600,0.683238,0.722008,0.687141
5,0.436900,0.734044,0.729730,0.692621
6,0.337100,0.877905,0.716216,0.685885
7,0.195300,1.076811,0.714286,0.677208
8,0.176700,1.353579,0.700772,0.675403



Early stopping triggered!
Best eval_f1: 0.6926


TrainOutput(global_step=1472, training_loss=0.5503257030378217, metrics={'train_runtime': 888.284, 'train_samples_per_second': 49.477, 'train_steps_per_second': 3.107, 'total_flos': 2.1832997713215984e+16, 'train_loss': 0.5503257030378217, 'epoch': 8.0})

In [ ]:
val_results = trainer.evaluate()
print(f"\nValidation Results:")
print(f"Accuracy: {val_results['eval_accuracy']:.4f}")
print(f"Macro F1: {val_results['eval_f1']:.4f}")

test_results = trainer.evaluate(test_tokenized)
print(f"\nTest Results:")
print(f"Accuracy: {test_results['eval_accuracy']:.4f}")
print(f"Macro F1: {test_results['eval_f1']:.4f}")

test_predictions = trainer.predict(test_tokenized)
test_preds = np.argmax(test_predictions.predictions, axis=-1)
test_pred_labels = [id2label[p] for p in test_preds]
y_true_test = test_df['clarity_label'].tolist()

print("\nClassification Report (Test Set):")
print(classification_report(y_true_test, test_pred_labels, labels=clarity_labels, zero_division=0))

In [ ]:
import os
import zipfile

os.makedirs("./predictions", exist_ok=True)

test_dataset_for_pred = test_tokenized.remove_columns(['clarity_label']) if 'clarity_label' in test_tokenized.column_names else test_tokenized
predictions = trainer.predict(test_dataset_for_pred)
pred_labels = np.argmax(predictions.predictions, axis=1)

label_map = {
    0: "Clear Reply",
    1: "Ambivalent", 
    2: "Clear Non-Reply"
}

prediction_filename = "./predictions/prediction"
with open(prediction_filename, 'w') as f:
    for pred in pred_labels:
        f.write(f"{label_map[pred]}\n")

zip_filename = "../predictions/prediction.zip"
with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
    zipf.write(prediction_filename, arcname="prediction")